# Building Data AI Agent — Production Engineering v2

This portfolio notebook explains the engineering contract behind the repository's **bounded LangGraph Text-to-SQL agent**.

It demonstrates why the project is more than `prompt → LLM → answer`:

`route → inspect schema → generate SQL → validate → execute tool → bounded repair → synthesize → trace → diagnose → evaluate`

**v2 additions:** component-level diagnosis, operational metrics, explicit stop policies, and secret-safe configuration.


## Architecture

```text
User Question
   ↓
Semantic Preflight
   ↓
READ_QUERY ───────────────┐
   ↓                      │
SQL Generation            │ WRITE / schema mismatch / ambiguous
   ↓                      ↓
AST + Allowlist Guardrails  Safe Policy Containment
   ↓
Read-only PostgreSQL
   ↓
Bounded one-retry repair (only on validation/execution failure)
   ↓
Grounded Answer
   ↓
Trace + Provenance + Diagnosis + Evaluation
```

Safety outcomes are not mislabeled as failures: a write request that never reaches database execution is **SAFE_POLICY_CONTAINMENT**.


## Secret-safe configuration

Credentials are never embedded in a publishable notebook. Use environment variables or a local `.env` excluded from Git.


In [ ]:
import os

print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("DATABASE_URL configured:", bool(os.getenv("DATABASE_URL")))


## Explicit loop / stopping policy

The production repository already bounds SQL repair. This model makes the broader control policy explicit and testable.


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class LoopPolicy:
    max_sql_retries: int = 1
    max_tool_failures: int = 1
    statement_timeout_ms: int = 5000
    max_result_rows: int = 200

def stop_reason(*, task_complete, retries, tool_failures, elapsed_ms, policy=LoopPolicy()):
    if task_complete:
        return "TASK_COMPLETE"
    if retries > policy.max_sql_retries:
        return "RETRY_LIMIT"
    if tool_failures > policy.max_tool_failures:
        return "TOOL_FAILURE_LIMIT"
    if elapsed_ms > policy.statement_timeout_ms:
        return "TIME_LIMIT"
    return "CONTINUE"


## Component-level failure diagnosis

Evaluation asks **whether** the run succeeded. Diagnosis asks **where the failure occurred and what should be fixed**.


In [ ]:
def diagnose_run(result, latency_budget_ms=25000.0):
    request_type = result.get("request_type", "UNKNOWN")
    validation = result.get("validation_status")
    execution = result.get("execution_status", "NOT_EXECUTED")
    retries = int(result.get("retry_count", 0) or 0)
    latency = float(result.get("latency_ms", 0.0) or 0.0)

    if latency > latency_budget_ms:
        return {"outcome": "LATENCY_BUDGET_EXCEEDED", "component": "LATENCY"}

    if request_type in {"WRITE_REQUEST", "SCHEMA_MISMATCH", "AMBIGUOUS"}:
        if execution == "NOT_EXECUTED":
            return {"outcome": "SAFE_POLICY_CONTAINMENT", "component": None}
        return {"outcome": "POLICY_CONTAINMENT_FAILURE", "component": "ROUTING_OR_POLICY"}

    if validation == "BLOCKED":
        return {
            "outcome": "REPAIR_EXHAUSTED" if retries else "SQL_VALIDATION_BLOCK",
            "component": "SQL_VALIDATION",
        }

    if execution == "ERROR":
        return {
            "outcome": "REPAIR_EXHAUSTED" if retries else "TOOL_EXECUTION_FAILURE",
            "component": "TOOL_EXECUTION",
        }

    if request_type == "READ_QUERY" and execution == "SUCCESS":
        return {"outcome": "SUCCESS", "component": None}

    return {"outcome": "UNKNOWN_OUTCOME", "component": "UNKNOWN"}


In [ ]:
demo_runs = [
    {
        "request_type": "READ_QUERY",
        "validation_status": "VALID",
        "execution_status": "SUCCESS",
        "retry_count": 0,
        "latency_ms": 850,
    },
    {
        "request_type": "WRITE_REQUEST",
        "execution_status": "NOT_EXECUTED",
        "retry_count": 0,
        "latency_ms": 2,
    },
    {
        "request_type": "READ_QUERY",
        "validation_status": "VALID",
        "execution_status": "ERROR",
        "retry_count": 1,
        "latency_ms": 1400,
    },
]

[diagnose_run(run) for run in demo_runs]


## Operational metrics contract

A production-oriented agent should expose:
- successful read completion rate
- safe policy-containment rate
- repair rate
- validation-block rate
- failure category distribution
- median / p95 latency
- result truncation rate
- trace depth

These metrics complement, rather than replace, semantic correctness and hold-out evaluation.


In [ ]:
from collections import Counter
from statistics import median

def summarize_runs(runs):
    diagnoses = [diagnose_run(r) for r in runs]
    outcomes = Counter(d["outcome"] for d in diagnoses)
    latencies = sorted(float(r.get("latency_ms", 0.0) or 0.0) for r in runs)

    return {
        "runs": len(runs),
        "outcome_categories": dict(outcomes),
        "repair_rate": sum(int(r.get("retry_count", 0) or 0) > 0 for r in runs) / len(runs),
        "latency_p50_ms": median(latencies),
        "traceable_failures": [d for d in diagnoses if d["component"] is not None],
    }

summarize_runs(demo_runs)


## Evaluation layers

| Layer | What is checked |
|---|---|
| Routing | Correct request class and safe containment |
| SQL safety | AST structure, table/column allowlists, write rejection |
| Execution | Read-only query completes within bounded runtime |
| Semantic result | Execution-equivalent result / answer contract |
| Repair | Recovery occurs within one retry, never an open loop |
| Observability | Request ID, trace, latency, retries, status |
| Provenance | Schema/SQL/result/ledger fingerprints |
| Regression | Previously passing cases remain passing |

The repository's frozen benchmark, unseen hold-out, safety tests, and container regression suite should remain the source of truth for reported scores.


## Why this is an AI Agent

A prompt wrapper:

`prompt → model → answer`

This project:

`question → route → inspect live schema → generate action → validate action → call database tool → observe → bounded repair → grounded synthesis → trace → diagnose → evaluate`

The LLM is therefore **one component inside a controlled execution harness**.

### Portfolio framing

**Production-oriented, bounded LangGraph data agent with read-only tool execution, SQL safety guardrails, structured tracing, component-level failure diagnosis, auditable provenance, and regression evaluation.**
